In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/crimes_2023_2026.csv",
    dtype={"IUCR": str, "FBI Code": str, "Beat": str},
    low_memory=False,
)
print(df.shape)
print(df.columns.tolist())
df.to_parquet("../data/processed/crimes_raw.parquet", index=False)

(940312, 22)
['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


In [4]:
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
print("Blank dates after parsing:", df["Date"].isna().sum())
print(df["Date"].min(), "->", df["Date"].max())

Blank dates after parsing: 0
2023-01-01 10:19:00 -> 2026-10-08 00:00:00


In [5]:
print(df["Date"].head())

0   2026-10-08
1   2026-10-08
2   2026-10-08
3   2026-10-08
4   2026-10-08
Name: Date, dtype: datetime64[us]


In [6]:
print("Missing values:\n", df.isna().sum(), "\n")
print("Crimes without coordinates:\n", df[df["Latitude"].isna()]["Primary Type"].value_counts().head(10), "\n")
print("Duplicate IDs:", df["ID"].duplicated().sum(), "\n")
print("Rows per year:\n", df.groupby(df["Date"].dt.year).size(), "\n")
print("Share of rows at exactly 00:00:", round((df["Date"].dt.hour.eq(0) & df["Date"].dt.minute.eq(0)).mean(), 3), "\n")
print("Overall arrest ratio:", round(df["Arrest"].mean(), 3), "\n")
print("Arrest ratio per crime type:\n", df.groupby("Primary Type")["Arrest"].mean().sort_values())

Missing values:
 ID                         0
Case Number                0
Date                       0
Block                      0
IUCR                       0
Primary Type               0
Description                0
Location Description    4751
Arrest                     0
Domestic                   0
Beat                       0
District                   0
Ward                       4
Community Area            36
FBI Code                   0
X Coordinate            7668
Y Coordinate            7668
Year                       0
Updated On                 0
Latitude                7668
Longitude               7668
Location                7668
dtype: int64 

Crimes without coordinates:
 Primary Type
DECEPTIVE PRACTICE            1721
THEFT                         1122
BATTERY                        878
CRIMINAL SEXUAL ASSAULT        639
OTHER OFFENSE                  540
CRIMINAL DAMAGE                422
OFFENSE INVOLVING CHILDREN     409
ASSAULT                        385
NARCOTIC